In [1]:
import glob
from pathlib import Path
import pandas as pd
import polars as pl

import aw
import job_search.jobs as jb
import job_search.config as conf
import job_search.scrape as sc
from job_search.config import P_RAW, P_INTERIM
from job_search.utils import now, reload
from job_search.scrape import DA_SF, DS_SF, DA_HEALTH, HEALTH

_now = now(time=False, days=0)
# _now = '2026-04-20'
P_raw_date = P_RAW / _now.replace('-', '/')
P_interim_date = P_INTERIM / _now.replace('-', '/')

In [2]:
glob_prefix = '../data/interim/2026/**'
health_df = sc.load_json_gz(glob_health := f'{glob_prefix}/{HEALTH}/*.json.gz', metadata=True)
ds_sf_df = sc.load_json_gz(glob_ds_sf := f'{glob_prefix}/{DS_SF}/*.json.gz', metadata=True)
da_health_df = sc.load_json_gz(glob_da_health := f'{glob_prefix}/{DA_HEALTH}/*.json.gz', metadata=True)
da_sf_df = sc.load_json_gz(glob_da_sf := f'{glob_prefix}/{DA_SF}/*.json.gz', metadata=True)

100%|██████████| 270/270 [00:03<00:00, 74.18it/s]


In [ ]:
health_df.shape[0], ds_sf_df.shape[0], da_health_df.shape[0], da_sf_df.shape[0]
# (8809, 8869, 4227, 6683)
# (14221, 17721, 4992, 7452)

(14221, 17721, 4992, 7452)

In [4]:
from markdownify import markdownify as md

def _process_df(jobs_df):
    jobs_df['_url'] = jb.JOB_HTTPS + jobs_df['requisition_id']
    jobs_df['_hash'] = jobs_df['requisition_id']
    jobs_df['_md'] = jobs_df['description']
    jobs_df = (jb.load_jobs(jobs_df)
        .sort_values('estimated_publish_date', ascending=False)
        .dropna(subset='_hash')
        .drop_duplicates(subset='_hash')
        .reset_index(drop=True)
    )#[[*jb.COLS, 'location_latitudes', 'location_longitudes']]
    jobs_df['_md'] = jobs_df['description'].fillna('').map(md)
    return jobs_df

In [6]:
## Takes ~10s
health_parquet_df = _process_df(health_df)
health_parquet_df.to_parquet(f'../data/cache/{HEALTH}.parquet')

In [7]:
## Takes ~10s
ds_sf_parquet_df = _process_df(ds_sf_df)
ds_sf_parquet_df.to_parquet(f'../data/cache/{DS_SF}.parquet')

In [8]:
## Takes ~10s
da_sf_parquet_df = _process_df(da_sf_df)
da_sf_parquet_df.to_parquet(f'../data/cache/{DA_SF}.parquet')

In [9]:
## Takes ~3s
da_health_parquet_df = _process_df(da_health_df)
da_health_parquet_df.to_parquet(f'../data/cache/{DA_HEALTH}.parquet')

In [10]:
all_df = (pd.concat([
    health_df,
    ds_sf_df,
    da_sf_df,
    da_health_df,
]).sort_values(['st_mtime', 'estimated_publish_date'], ascending=True)
    .dropna(subset='requisition_id')
    .drop_duplicates(subset='requisition_id')
    .reset_index(drop=True)
)
# all_df['st_mtime'].mean()
all_df.drop_duplicates(subset='requisition_id')
all_df

,st_mtime,st_size,requisition_id,job_id,board_token,source,apply_url,collapse_key,is_expired,title,...,company_latest_funding_type,company_latest_funding_year,company_latest_funding_amount,company_stock_exchange,company_stock_symbol,location_longitudes,location_latitudes,_url,_hash,_md
0,2026-04-21 01:07:28.288564682,529075,aeuj2gth21hdu1ng,rippling___aitp___d957f484-3943-4dcb-a0f9-72ef...,aitp,rippling,https://ats.rippling.com/aitp/jobs/d957f484-39...,ced7612c2db8707e93b8b32c22d1851175039200bcc225...,False,"Senior AI Engineer, Agentic Systems",...,Seed,2023.0,0.0,None,None,[],[],https://hiring.cafe/job/aeuj2gth21hdu1ng,aeuj2gth21hdu1ng,"<meta><p style=""font-family:""Basel Grotesk"",Ar..."
1,2026-04-21 01:07:28.288564682,529075,min2wb8by0uriqgk,grnhse___axle___4991731007,axle,grnhse,https://job-boards.greenhouse.io/axle/jobs/499...,5181ee3873b269f9cd26f881f54906a928e6145a3866a5...,False,Data Scientist II,...,Grant,2023.0,126000000.0,None,None,[],[],https://hiring.cafe/job/min2wb8by0uriqgk,min2wb8by0uriqgk,"<div style=""font-size: 10pt; font-family: 'Tah..."
2,2026-04-21 01:07:28.288564682,529075,h7vr5u38u76c6cca,workday___salesforce-wd12-external_career_site...,salesforce-wd12-external_career_site,workday,https://salesforce.wd12.myworkdayjobs.com/exte...,baaa53af7f5a95540dff68b9bf6ee996094687172f0126...,False,AI Forward Deployed Engineer (Senior/Lead/Prin...,...,None,NaN,NaN,NYSE,CRM,[-122.175],[47.6205],https://hiring.cafe/job/h7vr5u38u76c6cca,h7vr5u38u76c6cca,"<p style=""text-align:left""><i>To get the best ..."
3,2026-04-21 01:07:28.288564682,529075,xiwbb034q1ili6tb,jobvite___meltwater___ozwfzfwD,meltwater,jobvite,https://jobs.jobvite.com/meltwater/job/ozwfzfw...,b8a27ca174e1de9bec2f65e2c3b2ceb214cff68503ea35...,False,Senior/ Lead Data Solution Engineer,...,Private Equity,2023.0,65000000.0,None,None,[-122.2199],[37.4833],https://hiring.cafe/job/xiwbb034q1ili6tb,xiwbb034q1ili6tb,"Business ApplicationsHybrid Remote,\n\n ..."
4,2026-04-21 01:07:28.288564682,529075,jepny5coqv9pr5m1,ashby___fieldguide___fd1c14ef-a6bb-4cfe-a9fa-2...,fieldguide,ashby,https://jobs.ashbyhq.com/fieldguide/fd1c14ef-a...,fda3fe06d7c06312f3f9ad9af382ca6673c5743c6e2a7f...,False,Lead Data Scientist,...,Series C,2026.0,75000000.0,None,None,[-122.4194],[37.7749],https://hiring.cafe/job/jepny5coqv9pr5m1,jepny5coqv9pr5m1,"<p style=""min-height:1.5em""><strong>About Us:<..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12874,2026-06-05 21:46:38.398396969,160074,999qefv8xxo92isc,workday___wf-wd1-wellsfargojobs___senior-quant...,wf-wd1-wellsfargojobs,workday,https://wf.wd1.myworkdayjobs.com/wellsfargojob...,966eff2a9d8fae3e24bdfcd465a2890860d5f3b2456dc8...,False,Senior Quantitative Analytics Specialist (#001...,...,None,NaN,NaN,NYSE,WFC,[],[],https://hiring.cafe/job/999qefv8xxo92isc,999qefv8xxo92isc,None
12875,2026-06-05 21:46:38.398396969,160074,c53y15l2rwqvcny8,workday___lambweston-wd1-lamb_external___sr-da...,lambweston-wd1-lamb_external,workday,https://lambweston.wd1.myworkdayjobs.com/lamb_...,5257105112a73c6d35d44e9badf5434d789753edcda02d...,False,Sr Data Governance & MDM Specialist,...,None,NaN,NaN,NYSE,LW,[],[],https://hiring.cafe/job/c53y15l2rwqvcny8,c53y15l2rwqvcny8,None
12876,2026-06-05 21:46:38.398396969,160074,b1881nscye4t85ys,smartrecruiters___nbcuniversal3___be3f3d49-05f...,nbcuniversal3,smartrecruiters,https://jobs.smartrecruiters.com/NBCUniversal3...,6e95738bf527e67875fee8bc95b51e6eac5515c02da24c...,False,"Analyst, Digital Inventory",...,None,NaN,NaN,NASDAQ,CMCSA,[],[],https://hiring.cafe/job/b1881nscye4t85ys,b1881nscye4t85ys,None
12877,2026-06-05 21:46:56.002573252,196718,w7s126eifi3jyatq,workday___comscore-wd5-external___lead--client...,comscore-wd5-external,workday,https://comscore.wd5.myworkdayjobs.com/externa...,d97340d9ddee6d2b5ce12aee9bbc408ab1cf2802c0e76d...,False,"Lead, Client Success",...,None,NaN,NaN,NASDAQ,SCOR,[],[],https://hiring.cafe/job/w7s126eifi3jyatq,w7s126eifi3jyatq,None


In [11]:
all_parquet_df = (pd.concat([
        health_parquet_df,
        ds_sf_parquet_df,
        da_sf_parquet_df,
        da_health_parquet_df,
    ]).sort_values('estimated_publish_date', ascending=False)
        .dropna(subset='_hash')
        .drop_duplicates(subset='_hash')
        .reset_index(drop=True)
    )
all_df.to_parquet(f'../data/cache/ALL.parquet')

## Save JSON descriptions

In [ ]:
len([p for p in (jb.P_CACHE / 'json').iterdir()])
# 2402
# 3481

3481

In [15]:
import json

from tqdm import tqdm
import lxml.html

proxy = False
driver = sc.init_driver(proxy=proxy, headless=False)

errors = []
# for _hash in tqdm(all_df[all_df['description'].isna()].sort_values('st_mtime', ascending=False)['requisition_id']):
for _hash in tqdm(all_df.sort_values('st_mtime', ascending=False)['requisition_id']):
    # P_url = jb.P_CACHE / 'json' / f'2xztjhutpo56dvg9.json.gz'
    P_url = jb.P_CACHE / 'url' / f'{_hash}.html'
    P_json = jb.P_CACHE / 'json' / f'{_hash}.json.gz'
    if P_url.exists():
        continue

    # url = jb.VIEW_JOB_HTTPS + _hash
    url = jb.JOB_HTTPS + _hash
    print(url)

    # url_get_content = sc.selenium_get(url, driver=driver)
    try:
        url_get_content = sc.requests_get(url)
    except Exception:
        url_get_content = sc.selenium_get(url, driver=driver)

    root = lxml.html.fromstring(url_get_content)
    _next_data_list = root.xpath("//script[@id='__NEXT_DATA__']")
    if len(_next_data_list) == 0:
        jobs_dict = {}
        errors.append(url)
        print(url)
        continue
    else:
        _next_data = root.xpath("//script[@id='__NEXT_DATA__']")[0]
        jobs_dict = json.loads(_next_data.text_content())
        jobs_dict = jobs_dict.get('props', jobs_dict)

    sc.write_data(P_json, jobs_dict)
    with open(P_url, "w", encoding="utf-8") as f:
        f.write(url_get_content)
    sc.sleep(0.2, 0.5)

  0%|          | 0/12879 [00:00<?, ?it/s]

https://hiring.cafe/job/f4me995wq1kyqt5e
https://hiring.cafe/job/f4me995wq1kyqt5e
https://hiring.cafe/job/xvpkqw1qh27cp5dk
https://hiring.cafe/job/xvpkqw1qh27cp5dk
https://hiring.cafe/job/rgsgsg5songfyc45
https://hiring.cafe/job/rgsgsg5songfyc45
https://hiring.cafe/job/5uuqi289wlfg0nud
Writing: C:\Users\Alex\Dev\job-search\data\cache\json\5uuqi289wlfg0nud.json.gz


 31%|███▏      | 4050/12879 [00:00<00:00, 10949.10it/s]

https://hiring.cafe/job/vro5y7l7dwc0x1i5
Writing: C:\Users\Alex\Dev\job-search\data\cache\json\vro5y7l7dwc0x1i5.json.gz
https://hiring.cafe/job/76m0up8nuhs5lur4


 31%|███▏      | 4050/12879 [00:12<00:00, 10949.10it/s]

...

https://hiring.cafe/job/n7514nxtkfefc1e9
Writing: C:\Users\Alex\Dev\job-search\data\cache\json\n7514nxtkfefc1e9.json.gz


 32%|███▏      | 4157/12879 [02:54<2:29:40,  1.03s/it]

https://hiring.cafe/job/s3pkulmnbnaojhyx
Writing: C:\Users\Alex\Dev\job-search\data\cache\json\s3pkulmnbnaojhyx.json.gz
https://hiring.cafe/job/wsniam5htrb89g86
Writing: C:\Users\Alex\Dev\job-search\data\cache\json\wsniam5htrb89g86.json.gz
https://hiring.cafe/job/tli6dt4lgsv4sdg0
Writing: C:\Users\Alex\Dev\job-search\data\cache\json\tli6dt4lgsv4sdg0.json.gz
https://hiring.cafe/job/ijtees66qijuhusf
Writing: C:\Users\Alex\Dev\job-search\data\cache\json\ijtees66qijuhusf.json.gz
https://hiring.cafe/job/earbw5h4r7c9zhml
Writing: C:\Users\Alex\Dev\job-search\data\cache\json\earbw5h4r7c9zhml.json.gz
https://hiring.cafe/job/opw6rrshlg7l4403


 32%|███▏      | 4162/12879 [03:01<06:19, 22.95it/s]  


KeyboardInterrupt: 

In [371]:
url = 'https://hiring.cafe/job/2xztjhutpo56dvg9'

In [382]:
import json
import lxml.html

url_get_content = sc.requests_get(url)
root = lxml.html.fromstring(url_get_content)
_next_data_list = root.xpath("//script[@id='__NEXT_DATA__']")
if len(_next_data_list) == 0:
    jobs_dict = {}
else:
    _next_data = root.xpath("//script[@id='__NEXT_DATA__']")[0]
    jobs_dict = json.loads(_next_data.text_content())
    jobs_dict = jobs_dict.get('props', jobs_dict)

In [395]:
(P_json := jb.P_CACHE / 'json' / '2xztjhutpo56dvg9.json.gz').exists()

False

In [ ]:
## Takes ~35s
glob_str = '../data/cache/json/*.json.gz'
json_gz_paths = sorted([Path(path) for path in glob.glob(glob_str, recursive=True)])
# pl_df = pl.read_ndjson(json_gz_paths)
json_gz_list = []
for P_json_gz in tqdm(json_gz_paths):
    _json_gz_df = pd.read_json(P_json_gz, lines=True, compression='gzip')
    json_gz_list.append(_json_gz_df)
pl_df = pd.concat(json_gz_list)
_pageProps = pl_df['pageProps']

100%|██████████| 3114/3114 [00:21<00:00, 143.45it/s]


In [24]:
json_gz_df = pd.DataFrame({
    'hits': _pageProps,
})
json_gz_df['st_mtime'] = pd.to_datetime([p.stat().st_mtime for p in json_gz_paths], unit='s')
json_gz_df['st_size'] = [p.stat().st_size for p in json_gz_paths]
json_gz_df = json_gz_df[json_gz_df['hits'].str.len() == 1].reset_index(drop=True)

In [26]:
df = json_gz_df#.explode('hits')
job = df['hits'].str['job']
job_info = job.str['job_information']
v5_processed = job.str['v5_processed_job_data']
v5_company_data = job.str['v5_processed_company_data'].apply(lambda x: x if isinstance(x, dict) else {})
company_data = job.str['enriched_company_data']

df['requisition_id'] = job.str['requisition_id']
df['job_id'] = job.str['id']
df['board_token'] = job.str['board_token'].astype(str)
df['source'] = job.str['source']
df['apply_url'] = job.str['apply_url']
df['collapse_key'] = job.str['collapse_key']
df['is_expired'] = job.str['is_expired']

# Job information
df['title'] = job_info.str['title']
df['job_title_raw'] = job_info.str['job_title_raw']
df['description'] = job_info.str['description']
df['core_job_title'] = v5_processed.str['core_job_title']
df['requirements_summary'] = v5_processed.str['requirements_summary']

# Structured data
df['technical_tools'] = v5_processed.str['technical_tools']
df['licenses_certifications'] = v5_processed.str['licenses_or_certifications']
df['role_activities'] = v5_processed.str['role_activities']
df['language_requirements'] = v5_processed.str['language_requirements']

# Degree requirements
df['associates_degree_requirement'] = v5_processed.str['associates_degree_requirement']
df['associates_degree_fields'] = v5_processed.str['associates_degree_fields_of_study']
df['bachelors_degree_requirement'] = v5_processed.str['bachelors_degree_requirement']
df['bachelors_degree_fields'] = v5_processed.str['bachelors_degree_fields_of_study']
df['masters_degree_requirement'] = v5_processed.str['masters_degree_requirement']
df['masters_degree_fields'] = v5_processed.str['masters_degree_fields_of_study']
df['doctorate_degree_requirement'] = v5_processed.str['doctorate_degree_requirement']
df['doctorate_degree_fields'] = v5_processed.str['doctorate_degree_fields_of_study']
df['is_high_school_required'] = v5_processed.str['is_high_school_required']

# Experience requirements
df['min_industry_role_yoe'] = v5_processed.str['min_industry_and_role_yoe']
df['is_min_industry_role_yoe_not_mentioned'] = v5_processed.str['is_min_industry_and_role_yoe_not_mentioned']
df['min_management_leadership_yoe'] = v5_processed.str['min_management_and_leadership_yoe']
df['is_min_management_leadership_yoe_not_mentioned'] = v5_processed.str['is_min_management_and_leadership_yoe_not_mentioned']

# Role details
df['job_category'] = v5_processed.str['job_category']
df['commitment'] = v5_processed.str['commitment']
df['role_type'] = v5_processed.str['role_type']
df['seniority_level'] = v5_processed.str['seniority_level']

# Workplace
df['workplace_type'] = v5_processed.str['workplace_type']
df['workplace_physical_environment'] = v5_processed.str['workplace_physical_environment']
df['formatted_workplace_location'] = v5_processed.str['formatted_workplace_location']
df['is_workplace_worldwide_ok'] = v5_processed.str['is_workplace_worldwide_ok']
df['workplace_cities'] = v5_processed.str['workplace_cities']
df['workplace_counties'] = v5_processed.str['workplace_counties']
df['workplace_states'] = v5_processed.str['workplace_states']
df['workplace_countries'] = v5_processed.str['workplace_countries']
df['workplace_continents'] = v5_processed.str['workplace_continents']
df['boundless_workplace_states'] = v5_processed.str['boundless_workplace_states']
df['boundless_workplace_countries'] = v5_processed.str['boundless_workplace_countries']
df['boundless_workplace_continents'] = v5_processed.str['boundless_workplace_continents']
df['number_of_workplace_cities'] = v5_processed.str['number_of_workplace_cities']
df['number_of_workplace_counties'] = v5_processed.str['number_of_workplace_counties']
df['number_of_workplace_states'] = v5_processed.str['number_of_workplace_states']
df['number_of_workplace_countries'] = v5_processed.str['number_of_workplace_countries']
df['number_of_workplace_continents'] = v5_processed.str['number_of_workplace_continents']

# Work conditions
df['oral_communication_level'] = v5_processed.str['oral_communication_level']
df['physical_labor_intensity'] = v5_processed.str['physical_labor_intensity']
df['physical_position'] = v5_processed.str['physical_position']
df['computer_usage'] = v5_processed.str['computer_usage']
df['cognitive_demand'] = v5_processed.str['cognitive_demand']
df['air_travel_requirement'] = v5_processed.str['air_travel_requirement']
df['land_travel_requirement'] = v5_processed.str['land_travel_requirement']
df['morning_shift_work'] = v5_processed.str['morning_shift_work']
df['evening_shift_work'] = v5_processed.str['evening_shift_work']
df['overnight_work'] = v5_processed.str['overnight_work']
df['on_call_requirement'] = v5_processed.str['on_call_requirement']
df['weekend_availability_required'] = v5_processed.str['weekend_availability_required']
df['holiday_availability_required'] = v5_processed.str['holiday_availability_required']
df['overtime_required'] = v5_processed.str['overtime_required']

# Compensation
df['yearly_min_compensation'] = v5_processed.str['yearly_min_compensation']
df['yearly_max_compensation'] = v5_processed.str['yearly_max_compensation']
df['monthly_min_compensation'] = v5_processed.str['monthly_min_compensation']
df['monthly_max_compensation'] = v5_processed.str['monthly_max_compensation']
df['weekly_min_compensation'] = v5_processed.str['weekly_min_compensation']
df['weekly_max_compensation'] = v5_processed.str['weekly_max_compensation']
df['hourly_min_compensation'] = v5_processed.str['hourly_min_compensation']
df['hourly_max_compensation'] = v5_processed.str['hourly_max_compensation']
df['biweekly_min_compensation'] = v5_processed.str['bi-weekly_min_compensation']
df['biweekly_max_compensation'] = v5_processed.str['bi-weekly_max_compensation']
df['daily_min_compensation'] = v5_processed.str['daily_min_compensation']
df['daily_max_compensation'] = v5_processed.str['daily_max_compensation']
df['is_compensation_transparent'] = v5_processed.str['is_compensation_transparent']
df['listed_compensation_currency'] = v5_processed.str['listed_compensation_currency']
df['listed_compensation_frequency'] = v5_processed.str['listed_compensation_frequency']

# Benefits
df['four_oh_one_k_matching'] = v5_processed.str['401k_matching']
df['generous_paid_time_off'] = v5_processed.str['generous_paid_time_off']
df['four_day_work_week'] = v5_processed.str['four_day_work_week']
df['tuition_reimbursement'] = v5_processed.str['tuition_reimbursement']
df['retirement_plan'] = v5_processed.str['retirement_plan']
df['generous_parental_leave'] = v5_processed.str['generous_parental_leave']
df['fair_chance'] = v5_processed.str['fair_chance']
df['visa_sponsorship'] = v5_processed.str['visa_sponsorship']
df['relocation_assistance'] = v5_processed.str['relocation_assistance']
df['military_veterans'] = v5_processed.str['military_veterans']

# Other
df['security_clearance'] = v5_processed.str['security_clearance']
df['is_driver_license_required'] = v5_processed.str['is_driver_license_required']
df['position_employer_type'] = v5_processed.str['position_employer_type']
df['company_sector_and_industry'] = v5_processed.str['company_sector_and_industry']

# Dates
_estimated_publish_date = v5_processed.str['estimated_publish_date']
# if isinstance(_estimated_publish_date, int):
#     _estimated_publish_date = datetime.fromtimestamp(_estimated_publish_date / 1000)
df['estimated_publish_date'] = _estimated_publish_date
df['estimated_publish_date_millis'] = v5_processed.str['estimated_publish_date_millis']

_v5_is_public = v5_company_data.str['is_public']
# _v5_org_type = {True: 'Public', False: 'Private'}.get(_v5_is_public)
# if v5_company_data.get('is_non_profit'):
#     _v5_org_type = 'Non-Profit'
_v5_org_type = _v5_is_public.map({True: 'Public', False: 'Private'})
_v5_org_type.loc[v5_company_data.str['is_non_profit'].astype(bool)] = 'Non-Profit'

df = pd.concat([df,
    # User interactions
    job_info.str['hiddenFromUsers'].rename('hiddenFromUsers'),
    job_info.str['viewedByUsers'].rename('viewedByUsers'),
    job_info.str['applied_from_users'].rename('applied_from_users'),

    # Enriched Company data (embedded)
    company_data.str['enriched_at'].rename('company_enriched_at'),
    company_data.str['status'].rename('company_status'),
    company_data.str['name'].fillna(v5_processed.str['company_name']).fillna(v5_company_data.str['name']).rename('company_name'),
    company_data.str['homepage_uri'].fillna(v5_processed.str['company_website']).fillna(v5_company_data.str['website']).rename('company_homepage_uri'),
    company_data.str['hq_country'].fillna(v5_company_data.str['headquarters_country']).rename('company_hq_country'),
    company_data.str['parent_company'].fillna(v5_company_data.str['parent_company']).rename('company_parent_company'),
    company_data.str['subsidiaries'].fillna(v5_company_data.str['subsidiaries']).rename('company_subsidiaries'),
    company_data.str['industries'].fillna(v5_company_data.str['industries']).rename('company_industries'),
    company_data.str['activities'].fillna(v5_processed.str['company_activities']).fillna(v5_company_data.str['activities']).apply(lambda x: x if isinstance(x, list) else []).rename('company_activities'),
    company_data.str['nb_employees'].fillna(v5_company_data.str['number_employees'].astype(float)).rename('company_nb_employees'),
    company_data.str['year_founded'].fillna(v5_company_data.str['year_founded'].astype(float)).rename('company_year_founded'),
    company_data.str['tagline'].fillna(v5_processed.str['tagline']).fillna(v5_company_data.str['tagline']).rename('company_tagline'),
    company_data.str['organization_type'].fillna(_v5_org_type).rename('company_organization_type'),
    company_data.str['latest_funding_investors'].fillna(v5_company_data.str['investors']).rename('company_latest_funding_investors)'),
    company_data.str['latest_funding_type'].fillna(v5_company_data.str['latest_funding_series']).rename('company_latest_funding_type'),
    company_data.str['latest_funding_year'].fillna(v5_company_data.str['latest_funding_year'].astype(float)).rename('company_latest_funding_year'),
    company_data.str['latest_funding_amount'].fillna(v5_company_data.str['latest_funding_amount'].astype(float)).rename('company_latest_funding_amount'),
    company_data.str['stock_exchange'].fillna(v5_company_data.str['stock_exchange']).rename('company_stock_exchange'),
    company_data.str['stock_symbol'].fillna(v5_company_data.str['stock_exchange']).rename('company_stock_symbol'),

    # Extract location arrays
    job.str['_geoloc'].apply(lambda x: x if isinstance(x, list) else []).apply(lambda x_list: [x.get('lon') for x in x_list] if not isinstance(x_list, float) else []).rename('location_longitudes'),
    job.str['_geoloc'].apply(lambda x: x if isinstance(x, list) else []).apply(lambda x_list: [x.get('lat') for x in x_list] if not isinstance(x_list, float) else []).rename('location_latitudes'),
], axis=1).drop(columns='hits').reset_index(drop=True)

In [27]:
hash_set = set(df['requisition_id'])

In [28]:
x_df = df.sort_values(['estimated_publish_date', 'requisition_id']).reset_index(drop=True).drop(columns=['st_mtime', 'st_size', 'is_expired'])
y_df = all_df.query('requisition_id in @hash_set').sort_values(['estimated_publish_date', 'requisition_id']).reset_index(drop=True).drop(columns=['st_mtime', 'st_size', 'is_expired'])

In [29]:
x_df.shape, y_df.shape

((3109, 118), (3109, 121))

In [30]:
x_df.columns[:1]

Index(['requisition_id'], dtype='object')

In [31]:
# x_df[x_df.columns[118:]].equals(y_df[x_df.columns[118:]])

# x_df[x_df.columns[:9]].equals(y_df[x_df.columns[:9]])
x_df[x_df.columns[118:]].equals(y_df[x_df.columns[118:]])

True

In [32]:
eq_list = []
neq_list = []
for col in x_df.columns:
    if x_df[col].equals(y_df[col]):
        eq_list.append(col)
    else:
        neq_list.append(col)
eq_list = pd.Series(eq_list)
neq_list = pd.Series(neq_list)

In [33]:
len(eq_list), len(neq_list)

(58, 60)

In [34]:
x_df['hiddenFromUsers'] = x_df['hiddenFromUsers'].apply(lambda x: x if isinstance(x, list) else [])
y_df['hiddenFromUsers'] = y_df['hiddenFromUsers'].apply(lambda x: x if isinstance(x, list) else [])

In [35]:
nperc_list = []
for ncol in neq_list:
    if isinstance(x_df[ncol].iloc[0], list):
        nperc_list.append(sum(x_df[ncol].str.len() == y_df[ncol].str.len()))
    # elif isinstance(x_df[ncol].iloc[0], dict):
    #     nperc_list.append(sum(x_df[ncol].str.len() == y_df[ncol].str.len()))
    else:
        nperc_list.append(perc := sum(x_df[ncol] == y_df[ncol]))
    print(f'{ncol}: {perc/len(x_df):.2f}')


description: 0.00
technical_tools: 0.00
licenses_certifications: 0.00
role_activities: 0.00
language_requirements: 0.00
associates_degree_fields: 0.00
bachelors_degree_fields: 0.00
masters_degree_fields: 0.00
doctorate_degree_fields: 0.00
is_high_school_required: 1.00
is_min_management_leadership_yoe_not_mentioned: 1.00
commitment: 1.00
is_workplace_worldwide_ok: 1.00
workplace_cities: 1.00
workplace_counties: 1.00
workplace_states: 1.00
workplace_countries: 1.00
workplace_continents: 1.00
boundless_workplace_states: 1.00
boundless_workplace_countries: 1.00
boundless_workplace_continents: 1.00
number_of_workplace_cities: 1.00
number_of_workplace_counties: 1.00
number_of_workplace_states: 1.00
number_of_workplace_countries: 1.00
number_of_workplace_continents: 1.00
weekend_availability_required: 1.00
holiday_availability_required: 1.00
overtime_required: 1.00
monthly_min_compensation: 0.91
monthly_max_compensation: 0.91
weekly_min_compensation: 0.91
weekly_max_compensation: 0.91
hourly_

In [36]:
df_all = pd.concat([
    df,
    all_df.query('requisition_id not in @hash_set')
]).sort_values(['estimated_publish_date', 'requisition_id'], ascending=[False, True], ignore_index=True)
df_all

,st_mtime,st_size,requisition_id,job_id,board_token,source,apply_url,collapse_key,is_expired,title,...,company_latest_funding_type,company_latest_funding_year,company_latest_funding_amount,company_stock_exchange,company_stock_symbol,location_longitudes,location_latitudes,_url,_hash,_md
0,2026-06-04 23:50:14.735276222,4106,6t62p1uu48prt1bf,grnhse___weave___4274000009,weave,grnhse,https://job-boards.greenhouse.io/weave/jobs/42...,72495f30e8b417cc46adc426c9ec4cc2a8ba3206674273...,False,Senior AI Engineer,...,Series A,2025.0,2.000000e+07,None,None,[-122.4194],[37.7749],NaN,NaN,NaN
1,2026-06-04 23:47:18.206417322,5292,9u9hz2zrcbe557c5,grnhse___cafortune___5151678007,cafortune,grnhse,https://job-boards.greenhouse.io/cafortune/job...,4ad01afeb6d93eec58cf45449cc453138905a9cecf86be...,False,Senior Category Analyst,...,Corporate Round,2013.0,0.000000e+00,None,None,[-87.6298],[41.8781],NaN,NaN,NaN
2,2026-06-04 23:50:16.016179085,4248,7l6u9slrantdnlos,ashby___turquoise-health___db840fb4-7cf4-4b53-...,turquoise-health,ashby,https://jobs.ashbyhq.com/turquoise-health/db84...,9349c8e0a623b14761b26d2acb8ddf969dae44c68303ba...,False,"Data Science Engineer, Analytics",...,Series C,2026.0,4.000000e+07,None,None,[-117.1611],[32.7157],NaN,NaN,NaN
3,2026-06-04 23:48:31.926485062,4143,987nlzsi1as0amuk,breezy___pyrovio___a0653fb2684d,pyrovio,breezy,https://pyrovio.breezy.hr/p/a0653fb2684d-copil...,bfc344fe7010931c0b1caf788268c676e78945867ce349...,False,Copilot Developer/AI Engineer,...,None,NaN,NaN,None,None,[-81.519],[41.0814],NaN,NaN,NaN
4,2026-06-04 23:48:32.925237656,5762,bsm0p2eb45a8iwm3,grnhse___sonyinteractiveentertainmentglobal___...,sonyinteractiveentertainmentglobal,grnhse,https://job-boards.greenhouse.io/sonyinteracti...,34d49acddb204e381d887ddfce06a363e3abe31e854fc8...,False,"Senior Software Development Engineer In Test, ...",...,None,NaN,NaN,NYSE,SONY,[-122.3255],[37.563],NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12659,2026-05-08 22:03:08.651313305,110361,3dp4i78lijqp9dkx,icims___treliantllc___b38ffd47-59cd-46dc-bb8a-...,treliantllc,icims,https://careers-treliant.icims.com/jobs/1803/j...,c4fdab5f4c0db0f98deddc1f7ef85e6e1b012975a7d200...,False,"Consultant, Risk Management",...,Corporate Round,2025.0,NaN,None,None,[],[],https://hiring.cafe/job/3dp4i78lijqp9dkx,3dp4i78lijqp9dkx,"Overview:\n<p style=""margin: 0px;""><span style..."
12660,2026-05-08 22:03:08.651313305,110361,wbf5h3w5tz5zly01,taleo_careersection_s01stantec_230000P8,s01stantec,taleo_careersection,https://s01stantec.taleo.net/careersection/ex1...,cc10f9e7e974fb3278db4d585682047299deea5f615ebb...,False,Utility Economics & Financial Forecasting Analyst,...,None,NaN,NaN,New York Stock Exchange,STN,[],[],https://hiring.cafe/job/wbf5h3w5tz5zly01,wbf5h3w5tz5zly01,<p>Many of the world’s top engineers and scien...
12661,2026-05-08 22:03:08.651313305,110361,jxi8mno1sicbn9ks,icims___bottomline___118a1ead-c768-440f-8563-3...,bottomline,icims,https://careers-bottomline.icims.com/jobs/1471...,9d6296e83f0efd2f29f9367d5e63c75152c905ac230c30...,False,Director of Analytics and Reporting,...,None,NaN,NaN,None,None,[],[],https://hiring.cafe/job/jxi8mno1sicbn9ks,jxi8mno1sicbn9ks,"Overview & Benefits:\n<p style=""margin: 0px;"">..."
12662,2026-05-08 22:03:08.651313305,110361,5l6h3w5mb45sm6wc,breezy___vianai-systems___2709786e7a57,vianai-systems,breezy,https://vianai-systems.breezy.hr/p/2709786e7a5...,6380e5f1d8f422da23a203da1d7a74653ba07a9b256354...,False,Data Scientist,...,Series B,2021.0,1.400000e+08,None,None,[],[],https://hiring.cafe/job/5l6h3w5mb45sm6wc,5l6h3w5mb45sm6wc,<p><strong>The Opportunity:</strong></p> <p>Vi...


In [ ]:
## Takes ~35s
df_all_parquet_df = _process_df(df_all)
df_all_parquet_df.to_parquet(f'../data/cache/df.parquet')

In [47]:
(df_all_parquet_df['_md'] == '').sum()

np.int64(7)

## END